# 02. Neural Audio Codecs: EnCodec, RVQ, and Quantizer Dropout

- **Category:** 05_architectures / 05_generative_models
- **Lineage & Citations:** Neil Zeghidour et al., 2021 (*SoundStream*); Alexandre Défossez et al., 2022 (*EnCodec*); Rithesh Kumar et al., 2023 (*DAC*).
- **Purpose:** Formulates Multi-Scale Residual Vector Quantization (RVQ), Structured Quantizer Dropout, and Snake periodic activations to compress high-dimensional continuous audio waveforms into compact, discrete token streams.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Autoencoders VAE Beta VAE and VQ FSQ LFQ](01_autoencoders_vae_beta_vae_and_vq_fsq_lfq.ipynb) $\to$ `🔗 Downstream Syntheses:` [GANs DCGAN WGAN GP and StyleGAN Concepts](03_gans_dcgan_wgan_gp_and_stylegan_concepts.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**
> - **High-Dimensional Codebook Without Normalization:** Setting $D_{\text{code}} \ge 512$ without low-dimensional projection leads to severe codebook under-utilization (<10% active codewords).
> - **Unordered Quantizer Dropout:** Dropping arbitrary quantizer stages breaks the coarse-to-fine hierarchical residual structure.

## Mathematical Foundations & Governing Equations

### 1. Multi-Scale Residual Vector Quantization (RVQ)
Cascades $N_q$ codebooks where stage $k$ quantizes the residual $r_{k-1}$ left by stages $1 \dots k-1$:
$$r_0 = z, \quad i_k = \arg\min_j \|r_{k-1} - \mathcal{C}_{k, j}\|_2^2, \quad q_k = \mathcal{C}_{k, i_k}, \quad r_k = r_{k-1} - q_k$$
The reconstructed output with $n_{\text{active}}$ stages is:
$$\hat{z} = \sum_{k=1}^{n_{\text{active}}} q_k$$

### 2. Snake ($\sin^2$) Periodic Activation Function
$$\text{Snake}_\alpha(x) = x + \frac{1}{\alpha}\sin^2(\alpha x)$$

In [ ]:
import math
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/05_architectures/05_generative_models/02_neural_audio_codecs_encodec_rvq_and_quantizer_dropout.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)

z_in = weights["z_in"]
codebooks = weights["codebooks"]
alpha = float(weights["alpha"][0])
n_active = int(weights["n_active"][0])
print(f"Loaded {len(weights)} golden tensors on {env.device}, z_in: {list(z_in.shape)}, codebooks: {list(codebooks.shape)}")

In [ ]:
def snake_activation(x, alpha):
    return x + (1.0 / alpha) * torch.sin(alpha * x).pow(2)

def residual_vector_quantization(z, codebooks, n_active=None):
    N_q, K, D = codebooks.shape
    active = N_q if n_active is None else n_active
    residual = z.clone()
    z_q = torch.zeros_like(z)
    all_indices = []

    for k in range(N_q):
        cb = codebooks[k]
        dists = (
            residual.pow(2).sum(dim=-1, keepdim=True)
            - 2.0 * (residual @ cb.T)
            + cb.pow(2).sum(dim=-1).unsqueeze(0)
        )
        idx = torch.argmin(dists, dim=-1)
        all_indices.append(idx)
        quantized_k = cb[idx]

        if k < active:
            z_q = z_q + quantized_k
            residual = residual - quantized_k
        else:
            residual = residual - quantized_k

    return z_q, torch.stack(all_indices, dim=0), residual

In [ ]:
# Quantitative Mathematical Assertions & Golden Parity Verification
s_out = snake_activation(weights["x_audio"], alpha)
err_snake = float(torch.max(torch.abs(s_out - weights["snake_out"])))
assert err_snake < 1e-6, f"Snake activation error: {err_snake:.2e}"

z_q_out, idx_out, res_out = residual_vector_quantization(z_in, codebooks, n_active=n_active)
err_zq = float(torch.max(torch.abs(z_q_out - weights["z_q"])))
err_idx = float(torch.max(torch.abs(idx_out.float() - weights["indices"])))
err_res = float(torch.max(torch.abs(res_out - weights["residual"])))
assert err_zq < 1e-6, f"RVQ z_q error: {err_zq:.2e}"
assert err_idx < 1e-6, f"RVQ indices error: {err_idx:.2e}"
assert err_res < 1e-6, f"RVQ residual error: {err_res:.2e}"
print(f"Parity Verified: Snake={err_snake:.2e}, z_q={err_zq:.2e}, indices={err_idx:.2e}, residual={err_res:.2e}")

In [ ]:
import matplotlib.pyplot as plt
from core.benchmark import benchmark_fn, format_benchmark_table

b_snake = benchmark_fn(lambda: snake_activation(weights["x_audio"], alpha), iters=50, name="Snake Activation")
b_rvq = benchmark_fn(lambda: residual_vector_quantization(z_in, codebooks, n_active=n_active), iters=50, name="RVQ Quantization")
print(format_benchmark_table([b_snake, b_rvq]))

# Diagnostic Visualizations: Residual Energy Decay Across RVQ Stages
res_norms = []
curr_res = z_in.clone()
for k in range(codebooks.shape[0]):
    cb = codebooks[k]
    dists = curr_res.pow(2).sum(-1, keepdim=True) - 2.0 * (curr_res @ cb.T) + cb.pow(2).sum(-1).unsqueeze(0)
    idx = torch.argmin(dists, dim=-1)
    curr_res = curr_res - cb[idx]
    res_norms.append(float(torch.norm(curr_res).item()))

fig, ax = plt.subplots(1, 2, figsize=(8, 2.5), dpi=100)
x_plot = torch.linspace(-4, 4, 100)
ax[0].plot(x_plot.numpy(), snake_activation(x_plot, 2.0).numpy(), color="navy", label="Snake (alpha=2)")
ax[0].plot(x_plot.numpy(), x_plot.numpy(), "--", color="gray", alpha=0.6, label="Identity")
ax[0].set_title("Snake Periodic Activation")
ax[0].legend(fontsize=8)

ax[1].bar(range(1, len(res_norms) + 1), res_norms, color="teal")
ax[1].set_title("Residual L2 Norm vs RVQ Depth")
ax[1].set_xlabel("RVQ Stage")
ax[1].set_ylabel("Residual Norm")
plt.tight_layout()
plt.show()

In [ ]:
# Pathological Misuse vs Corrected Implementation: Random vs Prefix Quantizer Dropout
# Random dropout drops arbitrary quantizers (e.g. stage 0), destroying the base signal!
# Prefix dropout keeps stages 1..n, monotonically reducing error.
z_full, _, _ = residual_vector_quantization(z_in, codebooks, n_active=3)
z_prefix, _, _ = residual_vector_quantization(z_in, codebooks, n_active=1)
# Simulate random dropout by skipping stage 0 and using only stage 1
q1 = codebooks[1][torch.argmin(z_in.pow(2).sum(-1, keepdim=True) - 2.0 * (z_in @ codebooks[1].T) + codebooks[1].pow(2).sum(-1).unsqueeze(0), dim=-1)]
err_prefix = float(torch.norm(z_in - z_prefix))
err_random = float(torch.norm(z_in - q1))
print(f"Prefix Dropout Residual Error (Stage 1 only): {err_prefix:.4f}")
print(f"Random Dropout Error (Arbitrary stage skipped): {err_random:.4f}")
assert err_prefix < err_random, "Prefix dropout must outperform random stage skipping!"
print("Demonstrated requirement of contiguous prefix quantizer dropout.")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Coarse-to-Fine Quantization:** RVQ decomposes continuous representations into orthogonal hierarchical codebooks.
2. **Variable Bitrate via Prefix Dropout:** Structured prefix truncation enables multi-bandwidth streaming from a single network.
3. **Inductive Periodic Bias:** Snake activation prevents spectral distortion in sinusoidal waveform synthesis.

---
### Cross-Domain Generalisation Matrix
- **High-Fidelity Audio Compression:** EnCodec and DAC compress 44.1 kHz stereo audio into 6 kbps discrete tokens.
- **Language Model Sound Generation:** AudioCraft and MusicGen use RVQ tokens as the vocabulary for autoregressive transformers.
- **Image and Video Codecs:** Residual VQ scales to hierarchical latent diffusion and visual autoencoders.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [02_neural_audio_codecs_encodec_rvq_and_quantizer_dropout.md](../../../markdown/05_architectures/05_generative_models/02_neural_audio_codecs_encodec_rvq_and_quantizer_dropout.md)
- **PyTorch 2.x Module:** [02_neural_audio_codecs_encodec_rvq_and_quantizer_dropout.py](../../../pytorch2/05_architectures/05_generative_models/02_neural_audio_codecs_encodec_rvq_and_quantizer_dropout.py)
- **Functional JAX Module:** [02_neural_audio_codecs_encodec_rvq_and_quantizer_dropout.py](../../../jax/05_architectures/05_generative_models/02_neural_audio_codecs_encodec_rvq_and_quantizer_dropout.py)
- **Candle Rust Module:** [02_neural_audio_codecs_encodec_rvq_and_quantizer_dropout.rs](../../../candle/05_architectures/05_generative_models/02_neural_audio_codecs_encodec_rvq_and_quantizer_dropout.rs)
- **Hardware Kernel:** [02_neural_audio_codecs_encodec_rvq_and_quantizer_dropout.rs](../../../kernels/05_architectures/05_generative_models/02_neural_audio_codecs_encodec_rvq_and_quantizer_dropout.rs)
- **Universal Golden SafeTensors:** [02_neural_audio_codecs_encodec_rvq_and_quantizer_dropout.safetensors](../../../weights/05_architectures/05_generative_models/02_neural_audio_codecs_encodec_rvq_and_quantizer_dropout.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous Topic: 01. Autoencoders VAE Beta VAE and VQ FSQ LFQ](01_autoencoders_vae_beta_vae_and_vq_fsq_lfq.ipynb) | [Category Index](../../../README.md#05_architectures) | [Next Topic: 03. GANs DCGAN WGAN GP and StyleGAN Concepts](03_gans_dcgan_wgan_gp_and_stylegan_concepts.ipynb) &rarr;